In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# 局部 Fourier 状态码与 MULTI 共存验证 V1

Run all 执行固定三臂 OFF / PAYLOAD_MULTI / STATE_MULTI；两个标记臂在真实 Wan index25..49 控制，并继续完整原生 CFG/scheduler 生成。状态臂使用四个8x8局部块、非DC实 Fourier 正交分量、45个32-chip RM(1,5)状态、四条重叠age通道。状态目标总能量1；实际状态更新仅在L2超过1时缩小，不强制放大。载荷沿用原整帧FFT重复32bit；本轮不把它称为时间相关恢复。

这是新载体假设，不继承整帧FFT或旧空间DCT的存留结论。没有频带或强度扫描。实际局部空间支持和完整码本由公共key生成，不使用sample ID。每个物理时间位置联合承载延迟状态；相关重叠分量不作为独立票。

第一轮只保存3个源MP4（181帧，yuv420p），无派生裁剪或第二次压缩。每个视频以四个合法全局相位独立重编码，共12个VAE观测；所有相位均用相同44个规则潜窗口评分，额外行仅保存。接收器保留47520个局部状态代价、4176个有限零/单事件路径代价、24个相位/密钥文件。候选只在合法相位完整序列之间比较，不逐窗口拼接相位。75个写端诊断独立保存，不进入盲接收表。

末态、真相位局部排名、全相位有限路径排名、载荷和质量分别报告；没有自动方法PASS、校准存在检测或盲同步闭合结论。局部唯一不是最终同步充分条件，也不要求每个窗口独立唯一才研究序列。下一步接口为局部软观测、接收坐标、可用支持与合法相位分组；位置相关载荷、受扰片段及序列聚合尚待同链验证。

选择GPU后直接Run all。沿用当前Python及已可用Torch；实际依赖失败如实保留。Published source: `5d0284feecfe8e29bf9d1cc55afff2cf623d257d`.


In [ ]:
SOURCE_SHA = '5d0284feecfe8e29bf9d1cc55afff2cf623d257d'
FIXED = {'source_cases': 1, 'arms': 3, 'source_mp4': 3, 'derived_mp4': 0, 'normalized': 12, 'phase_path_searches': 24, 'phase_payload_reads': 24, 'primary_searches': 6, 'posthoc': 12, 'catalog_slots': 93960, 'valid_costs': 4176, 'structural_exclusions': 89784, 'local_state_costs': 47520, 'local_posthoc': 6, 'pilot_positives': 1, 'pilot_negatives': 5}
from pathlib import Path
import datetime,hashlib,json,os,subprocess,sys,traceback
STAMP=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=Path('/content/drive/MyDrive/Video-WM/Video-Local-Fourier-RM-V1')/STAMP
OUTPUT.mkdir(parents=True,exist_ok=False)
REPO=Path('/content/SC-SSTW-LOCAL-FOURIER-RM-'+STAMP)
RUN_OUTPUT=OUTPUT/'fixed_reference'
PYTHON=sys.executable
ARMS=('OFF','PAYLOAD_MULTI','STATE_MULTI');KEYS=('CORRECT','WRONG')
setup=dict(status='SETUP_STARTED',source_sha=SOURCE_SHA,fixed_denominator=FIXED,
 generation={a:{'status':'NOT_RUN_SETUP'} for a in ARMS},sources={a:{'status':'NOT_RUN_SETUP'} for a in ARMS},
 normalized={a+'/'+str(g):{'status':'NOT_RUN_SETUP'} for a in ARMS for g in range(4)},
 phase_reads={a+'/'+str(g)+'/'+k:{'status':'NOT_RUN_SETUP'} for a in ARMS for g in range(4) for k in KEYS},
 writer_diagnostics={a+'/'+str(i):{'status':'NOT_RUN_SETUP'} for a in ARMS for i in range(25,50)})
def write_json(path,value):
    temp=path.with_suffix(path.suffix+'.tmp');temp.write_text(json.dumps(value,indent=2)+'\n');os.replace(temp,path)
def failed(stage,exc):
    setup.update(status='SETUP_FAILED',stage=stage,error=f'{type(exc).__name__}: {exc}')
    write_json(OUTPUT/'setup_receipt.json',setup)
    write_json(OUTPUT/'setup_failure.json',dict(stage=stage,error=str(exc),traceback=traceback.format_exc()))
def logged(command,stage,cwd=None,check=True):
    try:
        with (OUTPUT/'execution.log').open('a') as log:
            log.write('COMMAND '+repr(command)+'\n');log.flush()
            child=subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
            for line in child.stdout:print(line,end='',flush=True);log.write(line);log.flush()
            code=child.wait();log.write('EXIT '+str(code)+'\n')
        if check and code:raise subprocess.CalledProcessError(code,command)
        return code
    except Exception as exc:failed(stage,exc);raise
write_json(OUTPUT/'setup_receipt.json',setup)
print('Fixed output:',OUTPUT)
if SOURCE_SHA is None:
    error=RuntimeError('Unpublished draft: bind verified published source SHA before Run all')
    failed('SOURCE_BINDING',error);raise error


In [ ]:
try:
    logged(['git','clone','--filter=blob:none','https://github.com/RICHAAARC/SC-SSTW.git',str(REPO)],'SOURCE_CLONE')
    logged(['git','-C',str(REPO),'fetch','origin',SOURCE_SHA],'SOURCE_FETCH')
    logged(['git','-C',str(REPO),'checkout','--detach',SOURCE_SHA],'SOURCE_CHECKOUT')
    actual=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
    dirty=subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True)
    if actual!=SOURCE_SHA or dirty:raise RuntimeError('source SHA/clean checkout mismatch')
    write_json(OUTPUT/'source_receipt.json',dict(source_sha=actual,clean=True))
    torch_ok=logged([PYTHON,'-c','import torch,torchvision; print(torch.__version__,torchvision.__version__)'],'TORCH_PROBE',check=False)==0
    if not torch_ok:
        logged([PYTHON,'-m','pip','install','--upgrade','torch','torchvision'],'TORCH_REPAIR')
    logged([PYTHON,'-m','pip','install','-r',str(REPO/'experiments/wan_state_clock/requirements-grow-video-reference.txt')],'DEPENDENCIES')
    dependency_code=logged([PYTHON,'-m','pip','check'],'DEPENDENCY_REPORT',check=False)
    setup['dependency_check_returncode']=dependency_code
    if dependency_code:print('Metadata conflicts retained; actual imports determine availability.')
    (OUTPUT/'environment_freeze.txt').write_text(subprocess.check_output([PYTHON,'-m','pip','freeze'],text=True))
    import_code='import json,torch; from diffusers import WanPipeline,AutoencoderKLWan,UniPCMultistepScheduler; from experiments.wan_state_clock.video_local_fourier_rm_run import environment; r=environment(); r.update(cuda_available=torch.cuda.is_available()); print(json.dumps(r))'
    env=json.loads(subprocess.check_output([PYTHON,'-c',import_code],cwd=REPO,text=True))
    write_json(OUTPUT/'environment_receipt.json',env)
    if not env['cuda_available']:print('CUDA unavailable: CPU path selected; much slower.')
    logged(['ffmpeg','-version'],'FFMPEG_PROBE');logged(['ffprobe','-version'],'FFPROBE_PROBE')
    setup.update(status='SETUP_COMPLETE',python_executable=PYTHON,environment_strategy='current_python_fresh_children')
    write_json(OUTPUT/'setup_receipt.json',setup)
except Exception as exc:failed('SOURCE_OR_ENVIRONMENT',exc);raise


In [ ]:
try:
    from google.colab import userdata
    token=userdata.get('HF_TOKEN')
    if token:os.environ['HF_TOKEN']=token
except Exception:
    pass
finally:
    token=None
command=[PYTHON,'-u','-m','experiments.wan_state_clock.video_local_fourier_rm_run','--output',str(RUN_OUTPUT)]
try:
    returncode=logged(command,'FIXED_RUN',cwd=REPO,check=False)
    RESULT_PATH=RUN_OUTPUT/'result.json'
    write_json(OUTPUT/'execution_receipt.json',dict(command=command,returncode=returncode,result_path=str(RESULT_PATH),result_exists=RESULT_PATH.is_file()))
    if not RESULT_PATH.is_file():raise RuntimeError('No runner result; full setup denominator is retained')
    setup.update(status='SUPERSEDED_BY_RESULT',result_path=str(RESULT_PATH));write_json(OUTPUT/'setup_receipt.json',setup)
except Exception as exc:failed('RUNNER',exc);raise


In [ ]:
result=json.loads(RESULT_PATH.read_text())
if result['source_sha']!=SOURCE_SHA or result['fixed_denominator']!=FIXED:raise RuntimeError('result identity mismatch')
print('Status:',result['status'],'Counts:',result['counts'])
print('Writer diagnostics:',result['writer_diagnostic_counts'])
for name,row in result['local_posthoc'].items():
    print('True-phase local:',name,row['status'],'unique rows=',row.get('unique_truth_rows'),'/44',
          'worst Delta=',row.get('worst_local_delta'),'path rank=',row.get('true_path_rank'),'path Delta=',row.get('true_path_delta'))
for name,row in result['searches'].items():
    print('Blind phase/path:',name,row['status'],'canonical=',row.get('canonical'),'ties=',len(row.get('top',[])))
for name,row in result['evaluations'].items():
    print(name,row['status'],'path matches=',row.get('canonical_path_matches'),'payload errors=',row.get('diagnostic_bit_errors'),
          'true-phase payload errors=',row.get('oracle_bit_errors'))
print('Quality:',result['quality'])
print('Failures:',result['failures'])
print(result['evidence_ceiling'])
print('Retained result:',RESULT_PATH)
from IPython.display import Video,display
for arm,row in result['sources'].items():
    print(arm,row['status'])
    if row['status']=='SAVED' and Path(row['path']).is_file():
        try:display(Video(str(row['path']),embed=True))
        except Exception as exc:print('Preview unavailable:',type(exc).__name__,str(exc))
